In [93]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial
from time import time

In [3]:
def attn(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    Y = np.empty_like(X)
    
    one_minus_gamma = 1.0 - gamma
    
    # Проходим по каждому сегменту, заданному границами
    for i in range(len(indices) - 1):
        start, end = indices[i], indices[i+1]
        if start == end:
            continue
            
        segment = X[start:end]
        n = len(segment)
        
        if gamma == 0:
            Y[start:end] = segment
        else:
            # Векторизованное вычисление EMA:
            # Y[t] = (1-gamma) * gamma^t * cumsum(X[t] * gamma^-t)
            powers = gamma ** np.arange(n)
            weighted = segment / powers
            cumweighted = np.cumsum(weighted)
            Y[start:end] = one_minus_gamma * powers * cumweighted
            
    return Y

In [4]:
def attn_vectorized(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    Y = np.empty_like(X)

    if gamma == 0:
        return X.copy()
    
    # Глобальные степени гаммы
    arange = np.arange(I)
    powers = gamma ** arange
    powers_inv = 1.0 / powers  # gamma ** -arange
    
    # Взвешенные значения для кумулятивной суммы
    weighted = X * powers_inv
    global_cum = np.cumsum(weighted)
    
    # Векторизованная сегментированная кумулятивная сумма
    # 1. Создаем маску сброса на началах сегментов (кроме первого)
    resets = indices[1:-1]
    mask = np.zeros(I, dtype=int)
    if len(resets) > 0:
        mask[resets] = resets
        
    # 2. Для каждой позиции находим индекс последнего сброса
    last_reset = np.maximum.accumulate(mask)
    
    # 3. Вычисляем величину коррекции (значение глобальной суммы перед сбросом)
    correction = np.zeros(I)
    valid = last_reset > 0
    correction[valid] = global_cum[last_reset[valid] - 1]
    
    # 4. Сегментированная сумма
    seg_cum = global_cum - correction
    
    # Финальный расчет EMA
    Y = (1.0 - gamma) * powers * seg_cum
    return Y

In [19]:
X = np.arange(20) + 1
Y = np.arange(20) + 2
Z = np.vstack((X, Y))
indices = [0, 20]
gamma = 0.1
beta = 0.5
Z

array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20],
       [ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21]])

In [11]:
attn(X, indices, gamma)

array([4.        , 6.4       , 7.84      , 4.        , 6.4       ,
       7.84      , 8.704     , 9.2224    , 9.53344   , 9.720064  ,
       4.        , 6.4       , 7.84      , 8.704     , 9.2224    ,
       9.53344   , 9.720064  , 9.8320384 , 9.89922304, 9.93953382])

In [12]:
attn_vectorized(X, indices, gamma)

array([4.        , 6.4       , 7.84      , 4.        , 6.4       ,
       7.84      , 8.704     , 9.2224    , 9.53344   , 9.720064  ,
       4.        , 6.4       , 7.84      , 8.704     , 9.2224    ,
       9.53344   , 9.720064  , 9.8320384 , 9.89922304, 9.93953382])

In [1]:
def attn(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    
    if gamma == 1.0:
        return X.copy()
    
    beta = 1.0 - gamma
    arange = np.arange(I)
    
    # P[t] = beta^t, P_inv[t] = beta^-t
    P = beta ** arange
    P_inv = 1.0 / P
    
    # Global weighted cumsum
    W = X * P_inv
    global_cum = np.cumsum(W)
    
    # Segmented cumsum via correction mask
    resets = indices[1:-1]
    mask = np.zeros(I, dtype=int)
    if len(resets) > 0:
        mask[resets] = resets
        
    last_reset = np.maximum.accumulate(mask)
    correction = np.zeros(I)
    valid = last_reset > 0
    correction[valid] = global_cum[last_reset[valid] - 1]
    
    seg_cum = global_cum - correction
    
    # Broadcast segment start values
    lengths = np.diff(indices)
    X_start = np.repeat(X[indices[:-1]], lengths)
    P_start_inv = np.repeat(P_inv[indices[:-1]], lengths)
    
    # Formula derived from recurrence: y_t = beta^t * (beta * x_start * beta^-start + gamma * seg_cum_t)
    Y = P * (beta * X_start * P_start_inv + gamma * seg_cum)
    
    return Y

In [9]:
attn(X, indices, gamma)

array([ 1.        ,  1.1       ,  1.29      ,  1.561     ,  1.9049    ,
        2.31441   ,  2.782969  ,  3.3046721 ,  3.87420489,  4.4867844 ,
       11.        , 11.1       , 11.29      , 11.561     , 11.9049    ,
       12.31441   , 12.782969  , 13.3046721 , 13.87420489, 14.4867844 ])

In [10]:
def attn(X, indices, gamma, beta):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros(I)
        valid = last_reset > 0
        correction[valid] = global_cum[last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[starts], lengths)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    Y = calc_right_ema(X, indices)
    
    X_rev = X[::-1]
    rev_indices = I - indices[::-1]
    Z_rev = calc_right_ema(X_rev, rev_indices)
    Z = Z_rev[::-1]
    
    return beta * Y + (1.0 - beta) * Z

In [35]:
print(attn(X, indices, gamma, beta=0.5))
print(attn(Y, indices, gamma, beta=0.5))
print(attn_2d(Z, indices, gamma, beta=0.5))
Z

[ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
  8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
 12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
 15.62542586 16.10788327]
[ 5.89211673  6.37457414  6.89452682  7.44664092  8.02593991  8.62774934
  9.24764488  9.88140314 10.52495476 11.17433922 11.82566078 12.47504524
 13.11859686 13.75235512 14.37225066 14.97406009 15.55335908 16.10547318
 16.62542586 17.10788327]
[[ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
   8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
  12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
  15.62542586 16.10788327]
 [ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
   8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
  12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
  15.62542586 16.10788327]]


array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20],
       [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20]])

In [131]:
def bidir_ema(X, 
            indices, 
            gamma: float, 
            beta: float = 0.5
        ) -> np.array:
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W, axis=1)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros((H, I))
        valid = last_reset > 0
        if np.any(valid):
            correction[:, valid] = global_cum[:, last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[:, starts], lengths, axis=1)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    Y = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    left_ema_rev = calc_right_ema(X_rev, rev_indices)
    left_ema = left_ema_rev[:, ::-1]
    
    return beta * Y + (1.0 - beta) * left_ema

print(bidir_ema(Z, indices, gamma))

[[   5.5      6.05     6.645 ... 4994.355 4994.95  4995.5  ]
 [   6.5      7.05     7.645 ... 4995.355 4995.95  4996.5  ]]


In [133]:
def bidir_ema(X, 
              indices, 
              gamma, 
              beta=0.5
             ) -> np.array:
    
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W, axis=1)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros((H, I))
        valid = last_reset > 0
        if np.any(valid):
            correction[:, valid] = global_cum[:, last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[:, starts], lengths, axis=1)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    right_ema = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    left_ema_rev = calc_right_ema(X_rev, rev_indices)
    left_ema = left_ema_rev[:, ::-1]
    
    return beta * right_ema + (1.0 - beta) * left_ema

print(bidir_ema(Z, indices, gamma))

[[   5.5      6.05     6.645 ... 4994.355 4994.95  4995.5  ]
 [   6.5      7.05     7.645 ... 4995.355 4995.95  4996.5  ]]


In [117]:
print(bidir_ema(Z, indices, gamma, beta=0.5))

[[   5.5      6.05     6.645 ... 4994.355 4994.95  4995.5  ]
 [   6.5      7.05     7.645 ... 4995.355 4995.95  4996.5  ]]


In [69]:
Z = np.vstack((X, Y))
print(attn_2d(Z, indices, gamma, beta=0.5))

[[ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
   8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
  12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
  15.62542586 16.10788327]
 [ 5.89211673  6.37457414  6.89452682  7.44664092  8.02593991  8.62774934
   9.24764488  9.88140314 10.52495476 11.17433922 11.82566078 12.47504524
  13.11859686 13.75235512 14.37225066 14.97406009 15.55335908 16.10547318
  16.62542586 17.10788327]]


In [77]:
from functools import partial
import jax
import jax.numpy as jnp

@partial(jax.jit, static_argnames=['gamma', 'beta'])
def attn_2d_jax(X: jax.Array, indices: jax.Array, gamma: float, beta: float) -> jax.Array:
    X = jnp.asarray(X)
    indices = jnp.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    alpha_pow = alpha ** jnp.arange(I)
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in: jax.Array, idx_in: jax.Array) -> jax.Array:
        W = X_in * alpha_inv_pow
        global_cum = jnp.cumsum(W, axis=1)
        
        resets = jnp.asarray(idx_in[1:-1], dtype=int).ravel()
        
        # Маска сброса через scatter (resets — гарантированно jax.Array)
        mask = jnp.zeros(I, dtype=int).at[resets].set(resets)
        last_reset = jnp.maximum.accumulate(mask)
        
        # Коррекция кумулятивной суммы
        valid = last_reset > 0
        safe_idx = jnp.where(valid, last_reset - 1, 0)
        correction_vals = global_cum[:, safe_idx]
        correction = jnp.where(valid[None, :], correction_vals, 0)
        
        seg_cum = global_cum - correction
        
        lengths = jnp.diff(idx_in)
        starts = idx_in[:-1]
        
        # Ключевое исправление: указываем total_repeat_length=I
        X_starts = jnp.repeat(X_in[:, starts], lengths, axis=1, total_repeat_length=I)
        alpha_inv_starts = jnp.repeat(alpha_inv_pow[starts], lengths, total_repeat_length=I)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    Y = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    Z_rev = calc_right_ema(X_rev, rev_indices)
    Z = Z_rev[:, ::-1]
    
    return beta * Y + (1.0 - beta) * Z

In [78]:
print(attn_2d_jax(Z, indices, gamma, beta=0.5))

[[ 4.8921156  5.3745728  5.8945255  6.4466395  7.025938   7.627748
   8.2476425  8.881402   9.524953  10.174337  10.82566   11.475044
  12.118597  12.752354  13.37225   13.974058  14.553358  15.105472
  15.625425  16.107883 ]
 [ 5.892115   6.3745728  6.894525   7.4466395  8.025937   8.627748
   9.2476425  9.881402  10.524953  11.174337  11.825659  12.475044
  13.118595  13.752354  14.37225   14.974058  15.553358  16.105473
  16.625423  17.107883 ]]


In [100]:
n = 5000
X = np.arange(n) + 1;
Y = np.arange(n) + 2;
Z = np.vstack((X, Y));
indices = [0, n/2, n];
gamma = 0.1;
beta = 0.5

In [106]:
print(jax.devices())
start_time = time()
for i in range(50):
    a = attn_2d_jax(Z + i, indices, gamma=0.1, beta=0.5)
print(time() - start_time)

[CudaDevice(id=0)]
16.44405722618103


In [104]:
start_time = time()
for i in range(50):
    a = attn_2d(Z + i, indices, gamma, beta=0.5)
print(time() - start_time)

0.021513938903808594


In [86]:
Z

array([[    1,     2,     3, ..., 19998, 19999, 20000],
       [    2,     3,     4, ..., 19999, 20000, 20001]], shape=(2, 20000))